# Dataset Parsing
Since the data provided is in JSON format, we first parsed it into a more manageable format such as CSV, in order to have the entire dataset in tabular form.


In [1]:
import ast
import pandas as pd
import pathlib
from datetime import datetime

number_of_csv = [f for f in pathlib.Path("./data/file_in_csv").glob("*.csv")]
number_of_json = [f for f in pathlib.Path("./data/file_in_json").glob("*.json")]


def translate_to_csv(dir_name: str):

    print(f"STEP 1 - Parsing DataSet")

    for file in number_of_json:

        print(f"Converting {file.name} from JSON to CSV")

        json = pd.read_json(f"./data/file_in_json/{file.name}")
        file_name = file.stem.split(".")[0]

        json.to_csv(f"./data/{dir_name}/{file_name}.csv")

    print(f"STEP 1 - Parsing DataSet Done")

# Dataset Cleaning and Data Flattening
After processing the data, we began analyzing it. In the code below, we extracted the potentially significant columns from the composite features (the ones indicated as 'Derived' in the document). Later on, we removed features with low variance or high sparsity of values. At this point, the dataset is ready for the EDA phase.


In [2]:
def modify_board_explainability(x: dict) -> int:
    """
    Calculates the average effective working time of an operator.

    This function parses 'boardExplainability' dictionary to extract the total effective time
    and divides it by the number of active operators to return a normalized metric.
    """

    time = x["totalEffectiveTime"]
    num_operators = (
        len(x["operatorExplanations"]) if x["operatorExplanations"] is not None else 0
    )

    return time / num_operators if time is not None and num_operators != 0 else 0


def modify_record(x: list, threshold_under: float, threshold_over: float) -> pd.Series:
    """
    Calculates the number of overworked operators based on a given threshold (chosen by us).

    Parses the 'record' feature to compute the mean 'fractionalBalance' for each operator.
    Returns the total count of operators whose average balance exceeds or stays below the given thresholds.
    """

    cont_over = 0
    cont_under = 0

    for dictionary in x:

        sum_ = 0

        if dictionary["fractionalBalance"] is not None:

            for dic in dictionary["fractionalBalance"]:

                if dic["fractional"] is not None:
                    sum_ += dic["fractional"]

            if sum_ / len(dictionary["fractionalBalance"]) > threshold_over:
                cont_over += 1
            elif sum_ / len(dictionary["fractionalBalance"]) < threshold_under:
                cont_under += 1

    return pd.Series({"numUnderworked": cont_under, "numOverworked": cont_over})


def extract_date(x: dict) -> pd.Series:
    """
    Extracts temporal features (day of the week and month) from a nested date object.
    """

    date = ast.literal_eval(str(x))["$date"]

    if date is None:
        return pd.Series(
            {
                "dayOfWeek": None,
                "Month": None,
                "planningDate_dt": None,
            }
        )

    planning_date = pd.to_datetime(date)
    day_of_week = planning_date.day_of_week
    month = planning_date.month

    return pd.Series(
        {
            "dayOfWeek": day_of_week,
            "Month": month,
            "planningDate_dt": planning_date,
        }
    )


def extract_forbidden_status(x: list):
    """
    Extracts logistical constraint metrics from patient records.

    Iterates through the 'forbiddenStatus' of each patient to calculate the total count,
    the average count per patient, and the total blocked time (in minutes) caused by
    mandatory appointments that were already scheduled.
    """

    if x is None:
        return pd.Series(
            {"totalForbidden": 0, "meanForbidden": 0, "sumForbiddenMinutes": 0}
        )

    sum_ = 0
    sum_minutes = 0

    for dictionary in x:

        forbidden = dictionary.get("forbiddenStatus")
        if forbidden is not None:
            visits_intervals = forbidden.get("visitsIntervals")
            if visits_intervals is not None:
                sum_ += len(visits_intervals)

                for visit in visits_intervals:

                    start = datetime.strptime(visit.get("start"), "%H:%M")
                    end = datetime.strptime(visit.get("end"), "%H:%M")
                    sum_minutes += (end - start).seconds // 60 if end > start else 0

    mean = sum_ / len(x) if len(x) > 0 else 0

    return pd.Series(
        {
            "totalForbidden": sum_,
            "meanForbidden": mean,
            "sumForbiddenMinutes": sum_minutes,
        }
    )


def extract_operator(lista: list) -> pd.Series:
    """
    Extracts and aggregates workforce capacity metrics from the operator list.

    This function uses a dictionary-based deduplication approach to isolate unique operators by their ID,
    preventing double-counting. Calculates total assignments, total and mean
    contractual working hours (TT), and shift distributions over the daily periods.
    """

    if lista is None:
        return pd.Series(
            {
                "totalAssignments": 0,
                "totalTT": 0,
                "meanTT": 0,
                "MorningOperators": 0,
                "AfternoonOperators": 0,
            }
        )

    total_assignments = 0
    total_tt = 0
    morning_operators = 0
    afternoon_operators = 0

    dict_operator = {}

    for dictionary in lista:

        operator = dictionary.get("operator")

        if operator is not None:

            operator_id = operator.get("id")

            if operator_id not in dict_operator:
                dict_operator[operator_id] = operator

    for op in dict_operator.values():

        total_assignments += op.get("assignments")
        total_tt += float(op.get("tt"))

        operating_times = op.get("operatingTimes")
        if operating_times is not None:
            morning_operators += 1 if operating_times[0] is not None else 0
            afternoon_operators += 1 if operating_times[1] is not None else 0

    mean_tt = total_tt / len(dict_operator) if len(dict_operator) > 0 else 0

    return pd.Series(
        {
            "totalAssignments": total_assignments,
            "totalTT": total_tt,
            "meanTT": mean_tt,
            "MorningOperators": morning_operators,
            "AfternoonOperators": afternoon_operators,
        }
    )


def extract_patient(lista: list) -> pd.Series:
    """
    Extracts and aggregates clinical and logistical features from the patient list.

    As the previous one, this function uses a dictionary-based deduplication strategy.
    It calculates metrics such as the total number of unique patients, occupied beds, unique rooms, patients
    requiring mobility aids, patients with operator preferences, and the total/mean
    minimum session lengths.
    """

    if lista is None:
        return pd.Series(
            {
                "numOccupiedBed": 0,
                "nonUniqueRoom": 0,
                "numPatientsWithNeeds": 0,
                "meanSessionMinLength": 0,
                "sumSessionMinLength": 0,
                "numPatientsWithPreferences": 0,
                "totalNumPatients": 0,
            }
        )

    total_occupied_beds = 0
    total_unique_rooms = set()
    total_patients_with_needs = 0
    total_min_session_length = 0
    total_patients_with_preferences = 0
    total_patients_in_group = 0

    dict_patients = {}

    for dictionary in lista:

        patient = dictionary.get("patient")

        if patient is not None:

            patient_id = patient.get("id")

            if patient_id not in dict_patients:
                dict_patients[patient_id] = patient

    total_patients = len(dict_patients)

    for patient in dict_patients.values():

        room = patient.get("room")
        if room is not None and str(room).strip() != "":
            total_unique_rooms.add(room)

        bed = patient.get("bed")
        if bed is not None and str(bed).strip() != "":
            total_occupied_beds += 1

        needs = patient.get("aidNeeds")
        if needs is not None and needs != "none":
            total_patients_with_needs += 1

        min_lenght = patient.get("overallMinLength")

        total_min_session_length += int(min_lenght) if min_lenght is not None else 0

        preferences = patient.get("preferredOps")
        if preferences is not None and len(preferences) > 0:
            total_patients_with_preferences += 1

        members = patient.get("members")
        if members is not None and len(members) > 0:
            total_patients_in_group += 1

    mean_session_length = (
        total_min_session_length / len(dict_patients) if len(dict_patients) > 0 else 0
    )

    return pd.Series(
        {
            "numOccupiedBed": total_occupied_beds,
            "numUniqueRoom": len(total_unique_rooms),
            "numPatientsWithNeeds": total_patients_with_needs,
            "meanSessionMinLength": mean_session_length,
            "sumSessionMinLength": total_min_session_length,
            "numPatientsWithPreferences": total_patients_with_preferences,
            "totalNumPatients": total_patients,
        }
    )


def extract_session(lista: list):
    """
    Extracts and aggregates operational and resource metrics from the session list.

    This function also uses a dictionary to deduplicate unique therapy sessions by ID. It calculates
    the total and mean ideal session lengths, along with the absolute count and
    average number of physical resources (equipment/spaces) required per single session.
    """

    if lista is None:
        return pd.Series(
            {
                "meanIdealSessionLength": 0,
                "sumIdealSessionLength": 0,
                "numResourcesNeeded": 0,
                "meanResourcesNeeded": 0,
            }
        )

    total_ideal_session_length = 0
    total_session_with_resources_needed = 0
    total_resources_needed = 0

    dict_sessions = {}

    for dictionary in lista:

        session = dictionary.get("session")

        if session is not None:

            session_id = session.get("id")

            if session_id not in dict_sessions:
                dict_sessions[session_id] = session

    for session in dict_sessions.values():
        ideal_lenght = session.get("idealLength")
        total_ideal_session_length += ideal_lenght if ideal_lenght is not None else 0

        resources = session.get("resources")
        if resources is not None:
            total_resources_needed += len(resources)
            total_session_with_resources_needed += 1

    mean_resources = (
        total_resources_needed / total_session_with_resources_needed
        if total_session_with_resources_needed != 0
        else 0
    )

    mean_ideal_session_length = (
        total_ideal_session_length / len(dict_sessions) if len(dict_sessions) > 0 else 0
    )

    return pd.Series(
        {
            "meanIdealSessionLength": mean_ideal_session_length,
            "sumIdealSessionLength": total_ideal_session_length,
            "numResourcesNeeded": total_resources_needed,
            "meanResourcesNeeded": mean_resources,
        }
    )


def extract_locations(x: list) -> pd.Series:
    """
    Extracts spatial utilization metrics from the nested macroLocations list.

    Calculates the total number of active macro-areas and utilizes a Set data structure
    to identify and count the exact number of unique individual workstations (micro-locations)
    used across the facility.
    """

    if x is None:
        return pd.Series({"numMacroLocationsUsed": 0, "numLocationsUsed": 0})

    num_macro_used = len(x)
    locations_unique = set()

    for macro in x:

        locations = macro.get("locations")

        for loc in locations:
            locations_unique.add(loc.get("id"))

    return pd.Series(
        {
            "numMacroLocationsUsed": num_macro_used,
            "numLocationsUsed": len(locations_unique),
        }
    )


def analyze_unique_columns(
    df: pd.DataFrame,
    column_name: str,
    nan_values_threshold: float,
    low_variance_threshold: float,
) -> bool:
    """
    Evaluates a specific DataFrame column to determine if it lacks predictive variance.

    Returns True if the column should be dropped due to low variance (only 1 unique value),
    high sparsity (> 90% missing values), or extreme imbalance (> 90% frequency for a single value).
    Returns False otherwise.
    """

    n_unique_values = df[column_name].nunique(dropna=False)
    n_null_values = df[column_name].isnull().sum() / len(df)

    if n_unique_values == 1 or n_null_values > nan_values_threshold:
        return True

    more_frequent_value = None

    try:
        more_frequent_value = df[column_name].value_counts(normalize=True).values[0]
    except TypeError:
        pass

    return (
        more_frequent_value is not None and more_frequent_value > low_variance_threshold
    )


def dataframe_engineering(df: pd.DataFrame) -> pd.DataFrame:

    df[["numMacroLocationsUsed", "numLocationsUsed"]] = df["macroLocations"].apply(
        lambda x: extract_locations(ast.literal_eval(x))
    )

    df["numUnassignedPatients"] = df["unassignedPatients"].apply(
        lambda x: len(ast.literal_eval(str(x)))
    )

    df[["numUnderworked", "numOverworked"]] = df["record"].apply(
        lambda x: modify_record(ast.literal_eval(x), -0.1, 0.1)
    )
    df.drop(columns="record", inplace=True)

    df[["dayOfWeek", "Month", "planningDate_dt"]] = df["planningDate"].apply(
        extract_date
    )

    df["agenda_list"] = df["agenda"].apply(lambda x: ast.literal_eval(str(x)))
    df["unassigned_list"] = df["unassignedPatients"].apply(
        lambda x: ast.literal_eval(str(x))
    )

    df["list_concat"] = df["agenda_list"] + df["unassigned_list"]

    df["totalTherapieDemand"] = df["list_concat"].apply(len)

    df[["totalForbidden", "meanForbidden", "sumForbiddenMinutes"]] = df[
        "list_concat"
    ].apply(lambda x: extract_forbidden_status(x))

    df[
        [
            "totalAssignments",
            "totalTT",
            "meanTT",
            "MorningOperators",
            "AfternoonOperators",
        ]
    ] = df["agenda_list"].apply(extract_operator)

    df[
        [
            "numOccupiedBed",
            "numUniqueRoom",
            "numPatientsWithNeeds",
            "meanSessionMinLength",
            "sumSessionMinLength",
            "numPatientsWithPreferences",
            "totalNumPatients",
        ]
    ] = df["list_concat"].apply(extract_patient)

    df[
        [
            "meanIdealSessionLength",
            "sumIdealSessionLength",
            "numResourcesNeeded",
            "meanResourcesNeeded",
        ]
    ] = df["list_concat"].apply(extract_session)

    columns_to_be_dropped = [
        "_id",
        "Unnamed: 0",
        "id",
        "__v",
        "lastUpdate",
        "lastUpdateUserId",
        "name",
        "version",
        "boardExplainability",
        "board",
        "teamId",
        "organization",
        "agenda_list",
        "unassigned_list",
        "agenda",
        "unassignedPatients",
        "list_concat",
        "macroLocations",
        "planningDate",
    ]

    for feature in df.columns:
        if feature not in columns_to_be_dropped and analyze_unique_columns(
            df, feature, 0.9, 0.9
        ):
            columns_to_be_dropped.append(feature)

    df.drop(columns=columns_to_be_dropped, inplace=True)

    return df


if __name__ == "__main__":

    if len(number_of_json) != len(number_of_csv):
        translate_to_csv("file_in_csv")

    dataframe_csv = [
        pd.read_csv(f"./data/file_in_csv/{f.name}")
        for f in pathlib.Path("./data/file_in_csv").glob("*.csv")
    ]

    df_master = pd.concat(dataframe_csv, ignore_index=True)

    final_df = dataframe_engineering(df_master)

    final_df.to_csv("./data/cleaned_dataset.csv")